In [1]:
##### Converts raw raster on population age groups into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # # of people aged (0-14) / # of people aged 15+

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from pathlib import Path
import rasterstats

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# raster directory 
raster_dir = Path(f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1") 

# save paths
pixels = f"{cd}/Data/Clean/Predictors/Rasters/child_dependency_ratio.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/child_dependency_ratio.csv"

In [4]:
#### Create age rasters 

# set file pattern
def make_pattern(code):
    return f"global_t_{code}_2020_CN_1km_R2025A_UA_v1.tif"

under_15_codes = ["00", "01", "05", "10", "15"]
over_15_codes  = ["20", "25", "30", "35", "40", "45", "50", "55", "60", "65", "70", "75", "80", "85", "90"]

# define function to sum rasters
def sum_rasters(codes, out_path):
    total = None
    meta  = None

    for code in codes:
        path = raster_dir / make_pattern(code)
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            nodata = src.nodata
            if meta is None:
                meta = src.meta.copy()

            # Mask nodata
            arr[arr == nodata] = np.nan

            if total is None:
                total = np.zeros_like(arr)

            # Add, treating nan as 0 (absent age group contributes nothing)
            total = np.nansum(np.stack([total, arr]), axis=0)

    # Where ALL inputs were nan, set back to nan
    all_nan = np.all(
        np.stack([
            np.isnan(rasterio.open(raster_dir / make_pattern(c)).read(1).astype(np.float32))
            for c in codes
        ]),
        axis=0
    )
    total[all_nan] = np.nan

    # Save
    meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
    out_arr = total.copy()
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(out_arr, 1)

    print(f"Saved: {out_path}  |  sum range: {np.nanmin(total):.2f} – {np.nanmax(total):.2f}")

sum_rasters(under_15_codes, raster_dir / "pop_under_15_2020.tif")
sum_rasters(over_15_codes,  raster_dir / "pop_over_15_2020.tif")

Saved: /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/pop_under_15_2020.tif  |  sum range: 0.00 – 43748.88
Saved: /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/pop_over_15_2020.tif  |  sum range: 0.00 – 84197.19


In [3]:
### Prep using target raster

### PREP 
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()
    ref_nodata    = ref.nodata
    ref_data      = ref.read(1)

# build reference valid mask: True where reference raster HAS data
if ref_nodata is not None:
    ref_valid = ~np.isnan(ref_data) if np.isnan(ref_nodata) else (ref_data != ref_nodata)
else:
    ref_valid = ~np.isnan(ref_data)

In [4]:
#### Run resampling for pixel scale 

# set paths
under_15 = f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/pop_under_15_2020.tif"
over_15 = f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/pop_over_15_2020.tif"

### RESAMPLE 
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

u_15  = resample_to_ref(under_15)
o_15 = resample_to_ref(over_15)


### CALCULATE 
# compute ag % GDP
with np.errstate(invalid="ignore", divide="ignore"):
    CDR = np.where(
        (o_15 > 0) & ~np.isnan(o_15) & ~np.isnan(u_15),
        u_15 / o_15,
        np.nan
    )

### ALIGN TO REFERENCE MASK

# Step 1: zero out pixels where reference has no data
CDR[~ref_valid] = np.nan

# Step 2: identify pixels that need filling
#   — reference has data, but predictor is still NaN
needs_fill = ref_valid & np.isnan(CDR)

print(f"Pixels needing fill: {needs_fill.sum()}")

if needs_fill.any():

    # reproject countries to match reference raster CRS
    if countries.crs != dst_crs:
        countries = countries.reindex(countries.index)  
        countries = countries.to_crs(dst_crs)

    # compute country-level mean of pct_ag (ignoring NaNs)
    # use rasterstats with the current (partial) predictor as an in-memory array
    country_stats = rasterstats.zonal_stats(
        countries,
        CDR,
        affine=dst_transform,
        stats=["mean"],
        nodata=np.nan,
    )

    # build a lookup: country index → mean pct_ag
    country_means = {
        i: s["mean"] for i, s in enumerate(country_stats)
        if s["mean"] is not None
    }

    # rasterise each country and fill the gap pixels with that country's mean
    fill_array = np.full(dst_shape, np.nan, dtype=np.float32)

    for idx, row in countries.iterrows():
        mean_val = country_means.get(idx)
        if mean_val is None:
            continue  # no data for this country at all; leave as NaN

        # burn this country's geometry into a boolean mask
        geom = [row.geometry]
        country_mask = geometry_mask(
            geom,
            transform=dst_transform,
            invert=True,          # True inside the country polygon
            out_shape=dst_shape,
        )

        # fill only the pixels that need filling AND fall inside this country
        to_fill = needs_fill & country_mask
        fill_array[to_fill] = mean_val

    # apply the fill
    CDR = np.where(needs_fill, fill_array, CDR)

    # report any pixels still unfilled (country had no valid data at all)
    still_missing = ref_valid & np.isnan(CDR)
    if still_missing.any():
        print(f"Warning: {still_missing.sum()} pixels still unfilled after country-average imputation "
              f"(country had no valid CDR data). These will be saved as NoData.")


### SAVE
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = CDR.copy()
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

Pixels needing fill: 839301


In [5]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_u15  = rasterstats.zonal_stats(gdf_proj, under_15, stats="sum", nodata=-9999)
zonal_o15 = rasterstats.zonal_stats(gdf_proj, over_15,   stats="sum", nodata=-9999)

### compute share at vector level
u15_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_u15])
o15_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_o15])

with np.errstate(invalid="ignore", divide="ignore"):
    CDR = np.where(
        (o15_sums > 0) & ~np.isnan(o15_sums) & ~np.isnan(u15_sums),
        u15_sums / o15_sums,
        np.nan
    )

result["child_dependency_ratio"] = CDR

### save
result.to_csv(vectors, index=False)